# Analyse Exploratoire des Données (EDA) — Cohorte FCCSS

**Projet** : Pôle Projet P15 (CentraleSupélec)  
**Objectif** : Prédiction des pathologies cardiaques sévères (grade ≥3 CTCAE) chez les survivants de cancers pédiatriques traités par radiothérapie thoracique.  
**Cohorte** : FCCSS (French Childhood Cancer Survivors Study) — 1 375 patients, 23 variables, prévalence 17.2%  
**Variable cible** : `Pathologie_cardiaque` (binaire)

---
## 0. Configuration et imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

# ------------------------------------------------------------------
# Constants
# ------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = Path('../../data/RT_Thorax_v1.csv')
FIG_DIR = Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'Pathologie_cardiaque'

# Variable groups
CONTINUOUS_VARS = ['age_diag', 'do_anthra_1K', 'Year_date_diag', 'mean', 'V5', 'V10', 'V15', 'V20']
BINARY_VARS = ['chimiotherapie_1K', 'anthra_1K', 'CT_sans_anthra']
CATEGORICAL_VARS = ['ctr', 'Year_date_diag_cut', 'age_diag_cut', 'iccc_type', 'gender']
EXCLUDED_VARS = ['numcent', 'deces', 'time', 'time_cut', 'age_exit', 'age_exit_cut']

# Publication palette
COLORS = {
    'blue': '#1f4e79',
    'green': '#2e8b57',
    'orange': '#c17c32',
    'purple': '#6a4c93',
}
COLOR_LIST = list(COLORS.values())
TARGET_COLORS = {0: COLORS['blue'], 1: COLORS['orange']}
TARGET_LABELS = {0: 'Pas de pathologie (0)', 1: 'Pathologie cardiaque (1)'}

# ------------------------------------------------------------------
# Publication-quality matplotlib settings
# ------------------------------------------------------------------
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def save_fig(fig, name):
    """Save figure to the figures directory."""
    path = FIG_DIR / name
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved: {path}')


print('Configuration OK')

---
## 1. Objectif

Cette analyse exploratoire vise a :

1. Comprendre la structure et la qualite des donnees de la cohorte FCCSS (1 375 patients).
2. Caracteriser la distribution de la variable cible `Pathologie_cardiaque` et identifier les desequilibres de classes.
3. Explorer les relations entre les variables cliniques, dosimetriques et la survenue de pathologies cardiaques.
4. Identifier les variables les plus pertinentes pour la modelisation predictive.
5. Detecter les anomalies, biais temporels et correlations fortes (notamment entre variables dosimetriques).

---
## 2. Chargement et qualite des donnees

**Objectif** : Charger le jeu de donnees, verifier sa structure, detecter les valeurs manquantes, doublons et anomalies connues.

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]} patients x {df.shape[1]} variables')
print(f'\nColonnes:\n{list(df.columns)}')

In [ ]:
# Data types overview
print('--- Types des variables ---')
print(df.dtypes.to_string())
print(f'\n--- Resume numerique ---')
df.describe().round(2)

In [ ]:
df.head(10)

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'N manquants': missing, '% manquants': missing_pct})
missing_df = missing_df[missing_df['N manquants'] > 0]

if len(missing_df) == 0:
    print('Aucune valeur manquante detectee.')
else:
    print('--- Valeurs manquantes ---')
    display(missing_df.sort_values('N manquants', ascending=False))

In [ ]:
# Duplicates
n_dup = df.duplicated().sum()
n_dup_id = df.duplicated(subset=['numcent']).sum()
print(f'Doublons (lignes completes): {n_dup}')
print(f'Doublons (numcent uniquement): {n_dup_id}')

In [ ]:
# Known anomalies check
print('--- Anomalie 1 : Year_date_diag_cut ---')
print('Valeurs uniques :', df['Year_date_diag_cut'].unique())
print()

# Check the suspicious value 3_1970-1980
mask_suspect = df['Year_date_diag_cut'] == '3_1970-1980'
if mask_suspect.any():
    suspect_years = df.loc[mask_suspect, 'Year_date_diag'].describe()
    print(f'Patients avec Year_date_diag_cut="3_1970-1980": {mask_suspect.sum()}')
    print(f'Leur Year_date_diag reel:\n{suspect_years}')
    print('\n=> ATTENTION : la valeur "3_1970-1980" est probablement un mislabel.')
    print('   Les annees reelles de ces patients correspondent a 1980-1990.')
else:
    print('Valeur "3_1970-1980" non trouvee (peut-etre deja corrigee).')

print('\n--- Anomalie 2 : iccc_type typo ---')
iccc_vals = df['iccc_type'].unique()
typos = [v for v in iccc_vals if 'nervouus' in str(v).lower()]
print(f'Valeurs avec typo "nervouus": {typos}')
if typos:
    print(f'Nombre de patients concernes: {df["iccc_type"].isin(typos).sum()}')

**Resultats** :
- Le jeu de donnees contient 1 375 patients et 23 variables.
- Deux anomalies connues sont confirmees : le mislabel de `Year_date_diag_cut` et la faute de frappe dans `iccc_type`.
- Ces anomalies seront prises en compte dans l'analyse mais ne sont pas corrigees ici pour preserver la tracabilite des donnees brutes.

---
## 3. Distribution de la variable cible

**Objectif** : Visualiser le desequilibre de classes de `Pathologie_cardiaque`.  
**Methode** : Diagramme en barres et diagramme circulaire.

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
target_pct = (target_counts / len(df) * 100).round(1)

print(f'Distribution de {TARGET}:')
for val in target_counts.index:
    print(f'  {TARGET_LABELS[val]}: {target_counts[val]} ({target_pct[val]}%)')
print(f'  Prevalence: {target_pct[1]}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# Bar chart
ax = axes[0]
bars = ax.bar(
    [TARGET_LABELS[i] for i in target_counts.index],
    target_counts.values,
    color=[TARGET_COLORS[i] for i in target_counts.index],
    edgecolor='white', linewidth=0.8, width=0.55
)
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f'{count}\n({pct}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Nombre de patients')
ax.set_title('Distribution de la variable cible')
ax.set_ylim(0, target_counts.max() * 1.2)

# Pie chart
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    target_counts.values,
    labels=[TARGET_LABELS[i] for i in target_counts.index],
    colors=[TARGET_COLORS[i] for i in target_counts.index],
    autopct='%1.1f%%', startangle=90,
    pctdistance=0.65, labeldistance=1.15,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight('bold')
ax.set_title('Repartition des classes')

fig.suptitle('Pathologie cardiaque severe (grade $\\geq$3 CTCAE)', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, '01_target_distribution.png')
plt.show()

**Interpretation** : Le desequilibre de classes est modere (82.8% vs 17.2%). Ce ratio est typique des etudes de toxicite tardive en oncologie pediatrique. Il faudra en tenir compte lors de la modelisation (stratification, metriques adaptees comme le ROC-AUC et le PR-AUC).

---
## 4. Distributions des variables continues

**Objectif** : Comparer les distributions des variables continues entre les deux classes.  
**Methode** : Histogrammes superposes par classe pour chaque variable continue.  
**Variables** : `age_diag`, `do_anthra_1K`, `Year_date_diag`, `mean`, `V5`, `V10`, `V15`, `V20`

In [ ]:
n_vars = len(CONTINUOUS_VARS)
n_cols = 4
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, var in enumerate(CONTINUOUS_VARS):
    ax = axes[i]
    for cls in [0, 1]:
        subset = df.loc[df[TARGET] == cls, var].dropna()
        ax.hist(
            subset, bins=30, alpha=0.55, density=True,
            color=TARGET_COLORS[cls], label=TARGET_LABELS[cls],
            edgecolor='white', linewidth=0.5
        )
    ax.set_title(var, fontweight='bold')
    ax.set_ylabel('Densite')
    if i == 0:
        ax.legend(fontsize=8, loc='upper right')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distributions des variables continues par classe', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
save_fig(fig, '02_continuous_distributions.png')
plt.show()

**Interpretation** :
- `mean` (dose moyenne au coeur) et les variables dosimetriques `V5`-`V20` montrent un decalage visible entre les deux classes : les patients avec pathologie cardiaque tendent a avoir recu des doses plus elevees.
- `Year_date_diag` : les patients diagnostiques plus tot semblent plus a risque (biais de suivi a explorer en section 10).
- `age_diag` : pas de separation nette entre les classes.
- `do_anthra_1K` : distribution tres desequilibree (beaucoup de zeros), ce qui rend l'histogramme moins informatif.

---
## 5. Boxplots comparatifs

**Objectif** : Comparer formellement les distributions des variables continues entre les deux classes.  
**Methode** : Boxplots par classe avec test de Mann-Whitney U (non-parametrique, adapte aux distributions non-normales). Affichage des p-values et des effectifs.

In [ ]:
n_vars = len(CONTINUOUS_VARS)
n_cols = 4
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, var in enumerate(CONTINUOUS_VARS):
    ax = axes[i]

    # Prepare data
    data_0 = df.loc[df[TARGET] == 0, var].dropna()
    data_1 = df.loc[df[TARGET] == 1, var].dropna()

    # Mann-Whitney U test
    stat, pval = stats.mannwhitneyu(data_0, data_1, alternative='two-sided')

    # Boxplot
    bp = ax.boxplot(
        [data_0, data_1],
        tick_labels=[f'0 (n={len(data_0)})', f'1 (n={len(data_1)})'],
        patch_artist=True, widths=0.5,
        boxprops={'linewidth': 1.2},
        medianprops={'color': 'black', 'linewidth': 1.5},
        whiskerprops={'linewidth': 1.0},
        capprops={'linewidth': 1.0},
        flierprops={'markersize': 3, 'alpha': 0.5},
    )
    bp['boxes'][0].set_facecolor(TARGET_COLORS[0])
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor(TARGET_COLORS[1])
    bp['boxes'][1].set_alpha(0.6)

    # Significance annotation
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    pval_str = f'p={pval:.1e}' if pval < 0.001 else f'p={pval:.4f}'
    ax.set_title(f'{var}\n{pval_str} ({sig})', fontweight='bold', fontsize=10)
    ax.set_xlabel(TARGET)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots comparatifs avec test de Mann-Whitney U', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, '03_boxplots_improved.png')
plt.show()

**Interpretation** :
- Les variables dosimetriques (`mean`, `V5`-`V20`) montrent generalement des differences significatives entre les deux groupes.
- `Year_date_diag` est significativement different (les patients diagnostiques plus tot ont plus de pathologies, ce qui s'explique par un suivi plus long).
- `age_diag` : la difference est plus modeste, a analyser plus en detail avec les categories d'age.
- `do_anthra_1K` : p-value a interpreter avec precaution du fait de la distribution tres asymetrique.

---
## 6. Taux d'evenement par variables binaires et categorielles

**Objectif** : Mesurer le taux de pathologie cardiaque pour chaque modalite des variables binaires et categorielles.  
**Methode** : Barres empilees avec taux d'evenement et intervalles de confiance (Wilson).

### 6a. Variables binaires

In [ ]:
def wilson_ci(n_success, n_total, alpha=0.05):
    """Wilson score confidence interval for a binomial proportion."""
    if n_total == 0:
        return 0, 0, 0
    p_hat = n_success / n_total
    z = stats.norm.ppf(1 - alpha / 2)
    denom = 1 + z**2 / n_total
    center = (p_hat + z**2 / (2 * n_total)) / denom
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * n_total)) / n_total) / denom
    return p_hat, max(0, center - margin), min(1, center + margin)


fig, axes = plt.subplots(1, len(BINARY_VARS), figsize=(4 * len(BINARY_VARS), 5))
if len(BINARY_VARS) == 1:
    axes = [axes]

for ax, var in zip(axes, BINARY_VARS):
    categories = sorted(df[var].dropna().unique())
    rates, lowers, uppers, counts = [], [], [], []

    for cat in categories:
        mask = df[var] == cat
        n_total = mask.sum()
        n_event = df.loc[mask, TARGET].sum()
        rate, lo, hi = wilson_ci(n_event, n_total)
        rates.append(rate * 100)
        lowers.append(lo * 100)
        uppers.append(hi * 100)
        counts.append(n_total)

    x = np.arange(len(categories))
    yerr_lo = [r - l for r, l in zip(rates, lowers)]
    yerr_hi = [u - r for r, u in zip(rates, uppers)]

    bars = ax.bar(x, rates, color=COLORS['blue'], alpha=0.7, width=0.5,
                  edgecolor='white', linewidth=0.8)
    ax.errorbar(x, rates, yerr=[yerr_lo, yerr_hi], fmt='none',
                ecolor='black', capsize=4, capthick=1.2, linewidth=1.2)

    for xi, rate, count in zip(x, rates, counts):
        ax.text(xi, rate + max(yerr_hi) + 1.5, f'{rate:.1f}%\n(n={count})',
                ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in categories])
    ax.set_title(var, fontweight='bold')
    ax.set_ylabel("Taux d'evenement (%)")
    ax.set_ylim(0, max(rates) * 1.5 + 5)

    # Chi-2 test
    ct = pd.crosstab(df[var], df[TARGET])
    chi2, pval_chi2, _, _ = stats.chi2_contingency(ct)
    sig = '***' if pval_chi2 < 0.001 else '**' if pval_chi2 < 0.01 else '*' if pval_chi2 < 0.05 else 'ns'
    ax.text(0.5, 0.95, f'Chi2 p={pval_chi2:.4f} ({sig})',
            transform=ax.transAxes, ha='center', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

fig.suptitle("Taux d'evenement par variables binaires", fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, '04_event_rate_binary.png')
plt.show()

### 6b. Variables categorielles

In [ ]:
fig, axes = plt.subplots(len(CATEGORICAL_VARS), 1, figsize=(12, 5 * len(CATEGORICAL_VARS)))

for ax, var in zip(axes, CATEGORICAL_VARS):
    # Sort categories by event rate
    grouped = df.groupby(var)[TARGET].agg(['sum', 'count'])
    grouped['rate'] = grouped['sum'] / grouped['count']
    grouped = grouped.sort_values('rate', ascending=False)

    categories = grouped.index.tolist()
    rates, lowers, uppers, counts = [], [], [], []

    for cat in categories:
        n_total = grouped.loc[cat, 'count']
        n_event = grouped.loc[cat, 'sum']
        rate, lo, hi = wilson_ci(int(n_event), int(n_total))
        rates.append(rate * 100)
        lowers.append(lo * 100)
        uppers.append(hi * 100)
        counts.append(int(n_total))

    x = np.arange(len(categories))
    yerr_lo = [r - l for r, l in zip(rates, lowers)]
    yerr_hi = [u - r for r, u in zip(rates, uppers)]

    # Color bars by rate (gradient)
    norm_rates = np.array(rates)
    if norm_rates.max() > norm_rates.min():
        norm_vals = (norm_rates - norm_rates.min()) / (norm_rates.max() - norm_rates.min())
    else:
        norm_vals = np.ones_like(norm_rates) * 0.5
    bar_colors = [plt.cm.YlOrRd(0.3 + 0.5 * v) for v in norm_vals]

    bars = ax.bar(x, rates, color=bar_colors, edgecolor='white', linewidth=0.8, width=0.7)
    ax.errorbar(x, rates, yerr=[yerr_lo, yerr_hi], fmt='none',
                ecolor='black', capsize=3, capthick=1, linewidth=1)

    for xi, rate, count in zip(x, rates, counts):
        ax.text(xi, rate + max(yerr_hi) + 0.8, f'{rate:.1f}%\n(n={count})',
                ha='center', va='bottom', fontsize=8)

    # Baseline
    overall_rate = df[TARGET].mean() * 100
    ax.axhline(y=overall_rate, color='red', linestyle='--', linewidth=1, alpha=0.7,
               label=f'Prevalence globale ({overall_rate:.1f}%)')

    ax.set_xticks(x)
    ax.set_xticklabels([str(c)[:30] for c in categories], rotation=45, ha='right', fontsize=8)
    ax.set_title(f"{var}", fontweight='bold')
    ax.set_ylabel("Taux d'evenement (%)")
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, max(rates) * 1.4 + 5)

    # Chi-2 test
    ct = pd.crosstab(df[var], df[TARGET])
    chi2, pval_chi2, dof, _ = stats.chi2_contingency(ct)
    sig = '***' if pval_chi2 < 0.001 else '**' if pval_chi2 < 0.01 else '*' if pval_chi2 < 0.05 else 'ns'
    ax.text(0.98, 0.85, f'Chi2={chi2:.1f}, dof={dof}, p={pval_chi2:.4f} ({sig})',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

fig.suptitle("Taux d'evenement par variables categorielles", fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
save_fig(fig, '05_event_rate_categorical.png')
plt.show()

**Interpretation** :
- **Variables binaires** : `anthra_1K` (exposition aux anthracyclines) et `chimiotherapie_1K` montrent une association avec la pathologie cardiaque. Les anthracyclines sont un facteur de cardiotoxicite bien documente.
- **Variables categorielles** : 
  - `iccc_type` montre une heterogeneite importante du taux d'evenement selon le type de cancer.
  - `Year_date_diag_cut` confirme un gradient temporel (plus ancien = plus de pathologies).
  - `gender` : a analyser pour d'eventuelles differences.
  - `ctr` (centre de traitement) peut refleter des pratiques heterogenes.

---
## 7. Matrice de correlation

**Objectif** : Identifier les correlations entre variables numeriques, en particulier la multicolinearite entre les variables dosimetriques.  
**Methode** : Matrice de correlation de Pearson avec heatmap annotee.

In [ ]:
# Select numeric columns for correlation
numeric_cols = CONTINUOUS_VARS + BINARY_VARS + [TARGET]
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap, center=0,
    annot=True, fmt='.2f', square=True, linewidths=0.8,
    cbar_kws={'shrink': 0.8, 'label': 'Coefficient de Pearson'},
    ax=ax, vmin=-1, vmax=1,
    annot_kws={'fontsize': 8}
)

ax.set_title('Matrice de correlation (variables numeriques)', fontsize=14, fontweight='bold', pad=15)

fig.tight_layout()
save_fig(fig, '06_correlation_matrix.png')
plt.show()

In [ ]:
# Highlight dosimetric correlations
dosi_vars = ['mean', 'V5', 'V10', 'V15', 'V20']
dosi_corr = df[dosi_vars].corr()

print('--- Correlations entre variables dosimetriques ---')
print(dosi_corr.round(3).to_string())
print()

# Identify pairs with r > 0.85
high_corr_pairs = []
for i in range(len(dosi_vars)):
    for j in range(i + 1, len(dosi_vars)):
        r = dosi_corr.iloc[i, j]
        if abs(r) > 0.85:
            high_corr_pairs.append((dosi_vars[i], dosi_vars[j], r))

print(f'Paires avec |r| > 0.85 ({len(high_corr_pairs)} paires):')
for v1, v2, r in high_corr_pairs:
    print(f'  {v1} -- {v2}: r = {r:.3f}')

print('\n=> La variable `mean` capture l\'essentiel de l\'information dosimetrique.')
print('   Il est recommande d\'utiliser `mean` seule pour eviter la multicolinearite.')

**Interpretation** :
- Les variables dosimetriques `mean`, `V5`, `V10`, `V15`, `V20` sont tres fortement correlees entre elles (r > 0.85 pour la plupart des paires).
- `mean` seule capture l'essentiel de l'information dosimetrique. Il est preferable de ne conserver que `mean` en modelisation pour eviter la multicolinearite.
- Les correlations avec la cible (`Pathologie_cardiaque`) sont modestes, ce qui est attendu pour un probleme de classification binaire.

---
## 8. Analyse dosimetrique

**Objectif** : Explorer en detail la relation entre la dose moyenne au coeur (`mean`) et les volumes irradies (`V5`-`V20`).  
**Methode** : Pairplot des variables dosimetriques colore par classe cible.

In [ ]:
# Pairplot of dosimetric variables
dosi_df = df[dosi_vars + [TARGET]].copy()
dosi_df[TARGET] = dosi_df[TARGET].map(TARGET_LABELS)

palette = {TARGET_LABELS[0]: TARGET_COLORS[0], TARGET_LABELS[1]: TARGET_COLORS[1]}

g = sns.pairplot(
    dosi_df, hue=TARGET, palette=palette,
    diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15, 'edgecolor': 'none'},
    diag_kws={'linewidth': 1.5},
    height=2.2, aspect=1
)
g.figure.suptitle('Relations entre variables dosimetriques', fontsize=14, fontweight='bold', y=1.02)

save_fig(g.figure, '07_dosimetric_analysis.png')
plt.show()

**Interpretation** :
- La dose moyenne (`mean`) est tres fortement liee aux volumes irradies, comme attendu physiquement : une dose moyenne elevee implique qu'une grande proportion du volume cardiaque a recu des doses significatives.
- Les distributions KDE montrent un leger decalage pour les patients avec pathologie cardiaque vers des doses plus elevees.
- La relation `mean` vs `V20` est la plus informative cliniquement : elle quantifie la proportion du coeur recevant plus de 20 Gy.

---
## 9. Analyse par type de cancer (ICCC)

**Objectif** : Explorer la distribution des types de cancer (classification ICCC) et leur association avec la pathologie cardiaque.  
**Methode** : Distribution des types + taux d'evenement par type avec test du Chi-2.

In [ ]:
# ICCC type distribution and event rates
iccc_stats = df.groupby('iccc_type').agg(
    n_patients=(TARGET, 'count'),
    n_events=(TARGET, 'sum'),
).reset_index()
iccc_stats['event_rate'] = (iccc_stats['n_events'] / iccc_stats['n_patients'] * 100).round(1)

# Compute confidence intervals
ci_data = []
for _, row in iccc_stats.iterrows():
    rate, lo, hi = wilson_ci(int(row['n_events']), int(row['n_patients']))
    ci_data.append({'ci_lower': lo * 100, 'ci_upper': hi * 100})
iccc_stats = pd.concat([iccc_stats, pd.DataFrame(ci_data)], axis=1)
iccc_stats = iccc_stats.sort_values('event_rate', ascending=False)

print('--- Statistiques par type ICCC ---')
display(iccc_stats.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Panel 1: Distribution
ax = axes[0]
order = iccc_stats.sort_values('n_patients', ascending=False)['iccc_type']
counts_ordered = iccc_stats.set_index('iccc_type').loc[order, 'n_patients']
x = np.arange(len(order))
ax.bar(x, counts_ordered.values, color=COLORS['blue'], alpha=0.7, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([str(c)[:35] for c in order], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Nombre de patients')
ax.set_title('Distribution des types de cancer (ICCC)', fontweight='bold')
for xi, val in zip(x, counts_ordered.values):
    ax.text(xi, val + 3, str(val), ha='center', va='bottom', fontsize=8)

# Panel 2: Event rates
ax = axes[1]
order_rate = iccc_stats['iccc_type']
rates_ordered = iccc_stats.set_index('iccc_type').loc[order_rate]
x = np.arange(len(order_rate))

yerr_lo = rates_ordered['event_rate'] - rates_ordered['ci_lower']
yerr_hi = rates_ordered['ci_upper'] - rates_ordered['event_rate']

norm_rates = rates_ordered['event_rate'].values
if norm_rates.max() > norm_rates.min():
    norm_vals = (norm_rates - norm_rates.min()) / (norm_rates.max() - norm_rates.min())
else:
    norm_vals = np.ones_like(norm_rates) * 0.5
bar_colors = [plt.cm.YlOrRd(0.3 + 0.5 * v) for v in norm_vals]

ax.bar(x, rates_ordered['event_rate'].values, color=bar_colors, edgecolor='white')
ax.errorbar(x, rates_ordered['event_rate'].values,
            yerr=[yerr_lo.values, yerr_hi.values],
            fmt='none', ecolor='black', capsize=3, capthick=1)
ax.axhline(y=df[TARGET].mean() * 100, color='red', linestyle='--', linewidth=1, alpha=0.7,
           label=f'Prevalence globale ({df[TARGET].mean() * 100:.1f}%)')

ax.set_xticks(x)
ax.set_xticklabels([str(c)[:35] for c in order_rate], rotation=45, ha='right', fontsize=8)
ax.set_ylabel("Taux d'evenement (%)")
ax.set_title("Taux de pathologie cardiaque par type ICCC (trie par taux decroissant)", fontweight='bold')
ax.legend(fontsize=9, loc='upper right')

for xi, rate, n in zip(x, rates_ordered['event_rate'].values, rates_ordered['n_patients'].values):
    ax.text(xi, rate + max(yerr_hi.values) + 1, f'{rate:.0f}%\n(n={n})',
            ha='center', va='bottom', fontsize=7)

# Chi-2 test for ICCC
ct_iccc = pd.crosstab(df['iccc_type'], df[TARGET])
chi2_iccc, pval_iccc, dof_iccc, _ = stats.chi2_contingency(ct_iccc)
sig = '***' if pval_iccc < 0.001 else '**' if pval_iccc < 0.01 else '*' if pval_iccc < 0.05 else 'ns'
ax.text(0.98, 0.75, f'Chi2={chi2_iccc:.1f}, dof={dof_iccc}, p={pval_iccc:.4f} ({sig})',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

fig.suptitle('Analyse par type de cancer (ICCC)', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
save_fig(fig, '08_cancer_type_analysis.png')
plt.show()

**Interpretation** :
- Les lymphomes (type 02) constituent le groupe le plus represente dans la cohorte.
- Le taux de pathologie cardiaque varie considerablement selon le type de cancer, refletant des protocoles de traitement et des doses thoraciques differents.
- Le test du Chi-2 permet d'evaluer si cette heterogeneite est statistiquement significative.
- Note : la faute de frappe "nervouus" dans le type 04 est confirmee et devra etre corrigee en pre-traitement.

---
## 10. Biais temporel

**Objectif** : Analyser l'association entre la periode de diagnostic et le taux de pathologie cardiaque, en discutant le biais de survie.  
**Methode** : Taux d'evenement par periode avec IC, distribution du temps de suivi par periode, regression sur le taux d'evenement.

In [ ]:
# Event rate by Year_date_diag_cut
period_stats = df.groupby('Year_date_diag_cut').agg(
    n_patients=(TARGET, 'count'),
    n_events=(TARGET, 'sum'),
    mean_time=('time', 'mean'),
    median_time=('time', 'median'),
    mean_year=('Year_date_diag', 'mean'),
).reset_index()
period_stats['event_rate'] = (period_stats['n_events'] / period_stats['n_patients'] * 100).round(1)

# CI
ci_data = []
for _, row in period_stats.iterrows():
    _, lo, hi = wilson_ci(int(row['n_events']), int(row['n_patients']))
    ci_data.append({'ci_lower': lo * 100, 'ci_upper': hi * 100})
period_stats = pd.concat([period_stats, pd.DataFrame(ci_data)], axis=1)
period_stats = period_stats.sort_values('mean_year')

print('--- Statistiques par periode de diagnostic ---')
display(period_stats.round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

periods = period_stats['Year_date_diag_cut'].values
x = np.arange(len(periods))

# Panel 1: Event rate by period with CI
ax = axes[0]
rates = period_stats['event_rate'].values
yerr_lo = period_stats['event_rate'].values - period_stats['ci_lower'].values
yerr_hi = period_stats['ci_upper'].values - period_stats['event_rate'].values

ax.bar(x, rates, color=COLORS['orange'], alpha=0.7, edgecolor='white', width=0.6)
ax.errorbar(x, rates, yerr=[yerr_lo, yerr_hi], fmt='none',
            ecolor='black', capsize=5, capthick=1.2)
for xi, rate, n in zip(x, rates, period_stats['n_patients'].values):
    ax.text(xi, rate + max(yerr_hi) + 1, f'{rate:.1f}%\n(n={n})',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(periods, rotation=30, ha='right', fontsize=8)
ax.set_ylabel("Taux d'evenement (%)")
ax.set_title("Taux d'evenement par periode", fontweight='bold')
ax.axhline(y=df[TARGET].mean() * 100, color='red', linestyle='--', linewidth=1, alpha=0.7)
ax.set_ylim(0, max(rates) * 1.4 + 5)

# Panel 2: Follow-up time by period
ax = axes[1]
period_order = period_stats['Year_date_diag_cut'].values
time_data = [df.loc[df['Year_date_diag_cut'] == p, 'time'].dropna().values for p in period_order]

bp = ax.boxplot(
    time_data, tick_labels=period_order, patch_artist=True, widths=0.5,
    boxprops={'linewidth': 1},
    medianprops={'color': 'black', 'linewidth': 1.5},
    flierprops={'markersize': 2, 'alpha': 0.4},
)
for patch in bp['boxes']:
    patch.set_facecolor(COLORS['green'])
    patch.set_alpha(0.6)

ax.set_xticklabels(period_order, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Temps de suivi (annees)')
ax.set_title('Duree de suivi par periode', fontweight='bold')

# Panel 3: Event rate vs median follow-up
ax = axes[2]
ax.scatter(period_stats['median_time'], period_stats['event_rate'],
           s=period_stats['n_patients'] * 0.5, c=COLORS['purple'], alpha=0.7,
           edgecolors='white', linewidth=0.8)

for _, row in period_stats.iterrows():
    ax.annotate(
        str(row['Year_date_diag_cut']),
        (row['median_time'], row['event_rate']),
        textcoords='offset points', xytext=(8, 5), fontsize=7,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5)
    )

# Trend line
from numpy.polynomial.polynomial import polyfit
coeffs = polyfit(period_stats['median_time'], period_stats['event_rate'], 1)
x_line = np.linspace(period_stats['median_time'].min(), period_stats['median_time'].max(), 50)
y_line = coeffs[0] + coeffs[1] * x_line
ax.plot(x_line, y_line, '--', color='red', alpha=0.6, linewidth=1.2, label='Tendance lineaire')

# Spearman correlation
r_spear, p_spear = stats.spearmanr(period_stats['median_time'], period_stats['event_rate'])
ax.text(0.05, 0.95, f'Spearman r={r_spear:.2f}, p={p_spear:.3f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=8,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('Suivi median (annees)')
ax.set_ylabel("Taux d'evenement (%)")
ax.set_title('Taux vs suivi median (taille = N patients)', fontweight='bold')
ax.legend(fontsize=8)

fig.suptitle('Analyse du biais temporel', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, '09_temporal_trends.png')
plt.show()

### Explication du biais de survie

Le gradient temporel observe (taux d'evenement plus eleve pour les periodes anciennes) s'explique principalement par le **biais de survie** :

1. **Suivi plus long** : Les patients diagnostiques dans les annees 1960-1970 ont un recul de plus de 40 ans, ce qui leur laisse plus de temps pour developper une pathologie cardiaque tardive.

2. **Latence des effets cardiotoxiques** : Les effets de la radiotherapie sur le coeur se manifestent souvent 15 a 30 ans apres le traitement. Les cohortes recentes n'ont pas encore atteint cette fenetre de risque.

3. **Evolution des protocoles** : Les techniques de radiotherapie se sont ameliorees au fil du temps (meilleures protections cardiaques, doses plus precises), ce qui peut aussi contribuer a la reduction du taux.

4. **Biais de selection** : Les patients diagnostiques il y a longtemps et encore vivants aujourd'hui sont un sous-groupe biaise (les plus fragiles peuvent etre decedes avant d'etre inclus).

**Consequence pour la modelisation** : `Year_date_diag` est une variable confondante majeure. Son inclusion dans les modeles predictifs doit etre interpretee avec prudence.

---
## 11. Resume des observations cles

### Structure des donnees
- **1 375 patients**, 23 variables, **aucune valeur manquante**.
- Deux anomalies confirmees : mislabel dans `Year_date_diag_cut` ("3_1970-1980" devrait etre "3_1980-1990") et typo dans `iccc_type` ("nervouus").

### Variable cible
- Prevalence de **17.2%** (desequilibre modere). Stratification necessaire en validation croisee.

### Variables dosimetriques
- `mean`, `V5`, `V10`, `V15`, `V20` sont **tres fortement correlees** (r > 0.85).
- **`mean` seule** capture l'essentiel de l'information dosimetrique. Recommandation : utiliser uniquement `mean` en modelisation.
- Differences significatives entre les classes (Mann-Whitney, p < 0.001).

### Variables cliniques
- **Anthracyclines** (`anthra_1K`) : facteur de cardiotoxicite bien documente. Signal univarie faible en classification (Chi-2 p ~ 0.07) mais **confondant clinique majeur**, retenu comme ajustement (cf. notebook 02 et analyse de survie).
- **Type de cancer (ICCC)** : forte heterogeneite du risque selon le type (Chi-2 p < 0.001).
- **Annee de diagnostic** (`Year_date_diag`) : confondant temporel fort (cf. biais de survie ci-dessous).
- **Genre** : difference **non significative** (Chi-2 p ~ 0.50, AUC univarie ~ 0.48) ; **exclu** du modele final apres etude d'ablation (cf. notebook 02).

### Biais temporel
- **Gradient temporel fort** : les patients diagnostiques plus tot ont un taux d'evenement plus eleve.
- Ce gradient s'explique par le **biais de survie** (suivi plus long, latence cardiotoxique).
- `Year_date_diag` est une **variable confondante majeure**.

### Ensemble de variables candidat
Au terme de l'EDA, l'ensemble **candidat** transmis a la modelisation (notebook 02) est :
`mean`, `Year_date_diag`, `age_diag`, `iccc_type`, `anthra_1K` (+ `gender`, a confirmer).
Le protocole de selection du notebook 02 confirme les **5 premieres** variables et **exclut `gender`**.

### Performance attendue
- En **univarie**, le pouvoir discriminant plafonne autour de **ROC-AUC ~ 0.65-0.68** (la dose moyenne en tete).
- En **multivarie**, la combinaison des variables porte la performance a **ROC-AUC ~ 0.77** (Random Forest, notebook 02), coherent avec la complexite du phenomene et la taille de la cohorte.